In [ ]:
"""
Copyright (C) <2025>  <The Ohio State University>

This program is free software: you can redistribute it and/or modify it under
the terms of the GNU General Public License as published by the Free Software
Foundation, either version 3 of the License, or (at your option) any later version.
This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
See the GNU General Public License for more details. You should have received a copy of the
GNU General Public License along with this program.  If not, see <https://www.gnu.org/licenses/>
""";

### Process and merge raw data files to produce a supplementary data table.

In [ ]:
"""
loads and merges raw data files to produce a supplementary data table.
""";

In [ ]:
%reset -f
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
import numpy as np

In [ ]:
# --- Define helper for reverse complement ---
def reverse_complement(seq):
    """Returns the reverse-complement of a DNA sequence."""
    if not isinstance(seq, str): return seq
    complement_map = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N'}
    return "".join(complement_map.get(base, base) for base in reversed(seq))

def calculate_genomic_locus(df, use_start_end_columns=False, site_length=7):
    """
    Calculates strand-aware genomic coordinates for binding sites using the reference allele.
    Automatically chooses the correct coordinates based on indel type:
    - Deletions: uses Start/End (L-allele is reference)
    - Insertions: uses StartD/EndD (S-allele is reference)

    Parameters:
    - use_start_end_columns: If True, uses Start/End and StartD/EndD columns directly.
                            If False (default), uses BestinMotif/BestinMotifD strings.
    """
    is_forward = df['strand'] == '+'
    is_insertion = df['alt'].str.len() > df['ref'].str.len()
    is_deletion = df['ref'].str.len() > df['alt'].str.len()

    # Determine which allele is the reference
    l_allele_is_ref = is_deletion

    if use_start_end_columns:
        # Use the reference allele coordinates directly from the data columns
        # For deletions: use Start/End (L-allele coordinates)
        # For insertions: use StartD/EndD (S-allele coordinates)
        ref_start = np.where(l_allele_is_ref, df['Start'], df['StartD'])
        ref_end = np.where(l_allele_is_ref, df['End'], df['EndD'])
    else:
        # Extract relative coordinates from BestinMotif or BestinMotifD based on reference allele
        ref_start_str = np.where(l_allele_is_ref, df['BestinMotif'], df['BestinMotifD'])
        ref_start = pd.Series(ref_start_str, index=df.index).str.split(':').str[0].astype(int)
        #ref_start = pd.Series(ref_start_str).str.split(':').str[0].astype(int)

        # Calculate end coordinate (start + site_length - 1)
        ref_end = ref_start + site_length - 1

    # Calculate genomic coordinates using the working transformation logic
    ref_start_genomic = np.where(is_forward,
                                df['foldstart'] + ref_start - 1,
                                df['foldend'] - ref_start + 1)

    ref_end_genomic = np.where(is_forward,
                              df['foldstart'] + ref_end - 1,
                              df['foldend'] - ref_end + 1)

    # Ensure start is always the smaller coordinate
    start_final = np.minimum(ref_start_genomic, ref_end_genomic)
    end_final = np.maximum(ref_start_genomic, ref_end_genomic)

    return start_final, end_final

In [ ]:
# set working directory
working_directory = "/content/drive/MyDrive/work/data/RNA-protein/IndelPolymorphismsModulateDistalRNAProteinInteractions"
os.chdir(working_directory)

# define paths
coordinates_path = "dataHuman/sameSites/sequences.tsv"
results_path = "dataHuman/sameSites/KdDDG.txt"
vcf_path = "data/indels.vcf"
output_path = "dataHuman/sameSites/merged.csv"

In [ ]:
# laod and merge main data
coordinates_df = pd.read_csv(coordinates_path, sep='\s+')
coordinates_df.drop_duplicates(
    subset=['string', 'withdeltion', 'ref', 'alt', 'Start', 'End', 'StartD', 'EndD', "indelLoc"],
    inplace=True
)

results_df = pd.read_csv(results_path, sep='\s+')
merged_df = pd.merge(
    results_df, coordinates_df,
    left_on=['seq', 'withdeletion', 'ref', 'alt', 'start', 'end', 'startD', 'endD', "indelLoc"],
    right_on=['string', 'withdeltion', 'ref', 'alt', 'Start', 'End', 'StartD', 'EndD', "indelLoc"],
    how='left'
)

/tmp/ipython-input-6-2988064749.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  coordinates_df = pd.read_csv(coordinates_path, sep='\s+')


In [ ]:
# # Verify coordinate transformation using the new corrected function
# print("\n--- Running Sanity Checks ---")

# # Use the new function to calculate genomic coordinates
# start_final, end_final = calculate_genomic_locus(merged_df, use_start_end_columns=True)

# # Count mismatches
# start_mismatches = (start_final != merged_df['startinChrom']).sum()
# end_mismatches = (end_final != merged_df['endinChrom']).sum()

# print(f"Binding site start -> genomic: {len(merged_df) - start_mismatches}/{len(merged_df)} correct")
# print(f"Binding site end -> genomic: {len(merged_df) - end_mismatches}/{len(merged_df)} correct")

# # Show breakdown by indel type
# is_insertion = merged_df['alt'].str.len() > merged_df['ref'].str.len()
# is_deletion = merged_df['ref'].str.len() > merged_df['alt'].str.len()
# print(f"\nDeletions (L-allele is ref): {is_deletion.sum()}")
# print(f"Insertions (S-allele is ref): {is_insertion.sum()}")

# # Show simplified debug output - only mismatches
# start_mismatch_mask = start_final != merged_df['startinChrom']
# end_mismatch_mask = end_final != merged_df['endinChrom']
# any_mismatch_mask = start_mismatch_mask | end_mismatch_mask

# if any_mismatch_mask.any():
#     print(f"\n {any_mismatch_mask.sum()} rows with mismatches:")
#     debug_df = pd.DataFrame({
#         'Start_final': start_final[any_mismatch_mask],
#         'End_final': end_final[any_mismatch_mask],
#         'Actual_start': merged_df['startinChrom'][any_mismatch_mask],
#         'Actual_end': merged_df['endinChrom'][any_mismatch_mask]
#     })
#     #print(debug_df.head(20).to_string())  # Show first 10 mismatches
# else:
#     print("\nNo mismatches found!")
# print("")

In [ ]:
# Normalize Alleles in Main Data to Forward Strand
is_reverse_strand = merged_df['strand'] == '-'
merged_df.loc[is_reverse_strand, 'ref'] = merged_df.loc[is_reverse_strand, 'ref'].apply(reverse_complement)
merged_df.loc[is_reverse_strand, 'alt'] = merged_df.loc[is_reverse_strand, 'alt'].apply(reverse_complement)

# Load and Merge VCF Data
vcf_df = pd.read_csv(
    vcf_path, sep='\t', comment='#', header=None, usecols=[0, 1, 2, 3, 4],
    names=['vcf_chrom', 'vcf_pos', 'dbsnp_id', 'vcf_ref', 'vcf_alt']
)

# Handle multi-allelic variants by expanding them into separate biallelic records
multi_allelic_mask = vcf_df['vcf_alt'].str.contains(',', na=False)
if multi_allelic_mask.any():
    print(f"Found {multi_allelic_mask.sum()} multi-allelic variants. Expanding into biallelic records...")

    # Split multi-allelic rows and expand
    expanded_rows = []
    for _, row in vcf_df[multi_allelic_mask].iterrows():
        alts = row['vcf_alt'].split(',')
        for alt in alts:
            new_row = row.copy()
            new_row['vcf_alt'] = alt.strip()
            expanded_rows.append(new_row)

    # Combine with biallelic rows
    vcf_df = pd.concat([
        vcf_df[~multi_allelic_mask],  # Keep biallelic rows as-is
        pd.DataFrame(expanded_rows)    # Add expanded multi-allelic rows
    ], ignore_index=True)

    print(f"Expanded to {len(vcf_df)} total biallelic records.")

vcf_df = vcf_df.astype({'vcf_chrom': str, 'vcf_pos': int, 'vcf_ref': str, 'vcf_alt': str})
merged_df = merged_df.astype({'chrom': str, 'indelLocinChrom': int, 'ref': str, 'alt': str})

final_merged_df = pd.merge(
    merged_df, vcf_df,
    left_on=['chrom', 'indelLocinChrom', 'ref', 'alt'],
    right_on=['vcf_chrom', 'vcf_pos', 'vcf_ref', 'vcf_alt'],
    how='left'
).drop_duplicates(subset=['seq', 'withdeletion', 'ref', 'alt', 'start', 'end'])


/tmp/ipython-input-8-2927776909.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  vcf_df = pd.read_csv(


Found 127875 multi-allelic variants. Expanding into biallelic records...
Expanded to 1863010 total biallelic records.


In [ ]:
# The paper focuses on distal effects, so we filter out indels that directly modify the binding site.
bs_start, bs_end = calculate_genomic_locus(final_merged_df)

indel_start = final_merged_df['indelLocinChrom']
indel_end = indel_start + final_merged_df['vcf_ref'].str.len() - 1

# Keep rows where there is no overlap
has_no_overlap = ~((indel_start <= bs_end) & (bs_start <= indel_end))
merged_df_filtered = final_merged_df[has_no_overlap].copy()
print(f"Filtering out {len(final_merged_df) - len(merged_df_filtered)} rows where the indel overlaps a binding site.")

# Also filter out rows where the determined binding site is not fully within the binding region.
binding_site_start, binding_site_end = calculate_genomic_locus(merged_df_filtered)
binding_region_start, binding_region_end = merged_df_filtered['startinChrom'], merged_df_filtered['endinChrom']

is_contained = (binding_region_start <= binding_site_start) & (binding_site_end <= binding_region_end)
original_rows = len(merged_df_filtered)
merged_df_filtered = merged_df_filtered[is_contained].copy()
filtered_rows = len(merged_df_filtered)

if original_rows > filtered_rows:
    print(f"Filtering out {original_rows - filtered_rows} rows where the binding site is not fully within its binding region.")

# Filter out rows where the binding preference change is not physiologically significant.
k_BT = 0.616
original_rows_ddg = len(merged_df_filtered)
merged_df_filtered = merged_df_filtered[merged_df_filtered['ddG'].abs() > k_BT].copy()
filtered_rows_ddg = len(merged_df_filtered)

if original_rows_ddg > filtered_rows_ddg:
    print(f"Filtering out {original_rows_ddg - filtered_rows_ddg} rows where |ddG| <= k_BT ({k_BT}).")

Filtering out 17828 rows where the indel overlaps a binding site.
Filtering out 36524 rows where the binding site is not fully within its binding region.
Filtering out 363046 rows where |ddG| <= k_BT (0.616).


In [ ]:
# # --- Sanity Check: Verify Indel Coordinate Calculation ---
# is_forward_filt = merged_df_filtered['strand'] == '+'

# # Forward strand calculation
# fwd_calculated_indel_coord = merged_df_filtered['foldstart'] + merged_df_filtered['indelLoc'] - 1

# # Reverse strand calculation is conditional on indel type
# is_insertion_filt = merged_df_filtered['alt'].str.len() > merged_df_filtered['ref'].str.len()
# rev_calculated_indel_coord = np.where(
#     is_insertion_filt,
#     merged_df_filtered['foldend'] - merged_df_filtered['indelLoc'] + 1,
#     merged_df_filtered['foldend'] - merged_df_filtered['indelLoc'] - merged_df_filtered['size'] + 1
# )

# calculated_indel_coord = np.where(
#     is_forward_filt,
#     fwd_calculated_indel_coord, # Forward
#     rev_calculated_indel_coord    # Reverse
# )

# mismatch_mask = calculated_indel_coord != merged_df_filtered['indelLocinChrom']
# mismatches = mismatch_mask.sum()

# if mismatches == 0:
#     print("Sanity check passed: Strand-aware indel coordinate calculation matches source data perfectly.")
# else:
#     print(f"\nWARNING: Found {mismatches} mismatches between calculated and source indel coordinates. Review logic.")
#     mismatch_df = merged_df_filtered[mismatch_mask].copy()
#     mismatch_df['calculated_indel_coord'] = calculated_indel_coord[mismatch_mask]
#     debug_cols = ['chrom', 'strand', 'indelLocinChrom', 'calculated_indel_coord', 'foldstart', 'foldend', 'indelLoc', 'size']
#     print("First few mismatched rows for debugging:")
#     print(mismatch_df[debug_cols].head())
#     print("")

In [ ]:
 # --- Prepare Final Table ---
final_df = merged_df_filtered[['vcf_ref', 'vcf_alt', 'ddG', 'dbsnp_id']].copy()
final_df.rename(columns={'vcf_ref': 'ref_allele',
                         'vcf_alt': 'alt_allele',
                         'ddG': 'delta_delta_G',
                         'dbsnp_id': 'dbSNP_ID'}, inplace=True)


# extract coordinates of the binding site and binding region
binding_site_start, binding_site_end = calculate_genomic_locus(merged_df_filtered)
binding_region_start, binding_region_end = merged_df_filtered['startinChrom'], merged_df_filtered['endinChrom']

final_df['binding_site_locus'] = 'chr' + merged_df_filtered['chrom'].astype(str) + ':' + binding_site_start.astype(str) + '-' + binding_site_end.astype(str)
final_df['binding_region_locus'] = 'chr' + merged_df_filtered['chrom'].astype(str) + ':' + binding_region_start.astype(str) + '-' + binding_region_end.astype(str)
final_df['indel_genomic_coordinate'] = merged_df_filtered['indelLocinChrom']
final_df['chromosome'] = merged_df_filtered['chrom']

# --- Finalize and Save ---

final_df.insert(1, 'indel_locus', 'chr' + final_df['chromosome'].astype(str) + ':' + final_df['indel_genomic_coordinate'].astype(str))
final_df['delta_delta_G'] = final_df['delta_delta_G'].round(5)

# Reorder columns
final_columns = [
    'chromosome',
    'indel_genomic_coordinate',
    'dbSNP_ID',
    'ref_allele',
    'alt_allele',
    'binding_site_locus',
    'binding_region_locus',
    'delta_delta_G'
]

final_df = final_df[final_columns]
final_df.loc[:, 'delta_delta_G'] = final_df['delta_delta_G'].round(5)


# Sort the final table by the absolute magnitude of the effect
final_df['abs_ddG'] = final_df['delta_delta_G'].abs()
final_df.sort_values(by='abs_ddG', ascending=False, inplace=True)
final_df.drop(columns=['abs_ddG'], inplace=True)

final_df.drop_duplicates(inplace=True)

In [ ]:
# --- Sanity Check: Verify binding site is within the binding region ---
binding_site_start, binding_site_end = calculate_genomic_locus(merged_df_filtered)
binding_region_start, binding_region_end = merged_df_filtered['startinChrom'], merged_df_filtered['endinChrom']
is_contained = (binding_region_start <= binding_site_start) & (binding_site_end <= binding_region_end)

if is_contained.all():
    print("\nSanity check passed: All binding sites are correctly located within their binding regions.\n")
else:
    mismatches = len(is_contained) - is_contained.sum()
    print(f"\nWARNING: Found {mismatches} rows where the binding site is NOT within its binding region. Review logic.\n")
    # Optional: Display the mismatched rows for debugging
    # print(merged_df_filtered[~is_contained].head())



Sanity check passed: All binding sites are correctly located within their binding regions.



In [ ]:
# --- Save the Final Table ---
output_path = "dataHuman/sameSites/merged.csv"

final_df.to_csv(output_path, index=False)

### Add annotations to the supplementary table

In [ ]:
"""
Loads the supplementary table and annotates each row with Ensembl
transcript IDs using the local transcripts.fa.fai index file.
""";

In [ ]:
%reset -f
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os

In [ ]:
def parse_fai_to_intervals(fai_path):
    """
    Parses the custom transcripts.fa.fai file to create a dictionary
    mapping chromosomes to a list of transcript intervals.

    Returns:
        A dict where keys are chromosome numbers (e.g., '1', 'X') and
        values are lists of tuples, with each tuple being
        (start_coord, end_coord, strand, transcript_id).
    """
    transcript_intervals = {}
    print(f"Reading transcript data from: {fai_path}")
    with open(fai_path, 'r') as f:
        for line in f:
            parts = line.split()
            name_parts = parts[0].split('|')

            # Extract data from the sequence name
            transcript_id = name_parts[0]
            strand = name_parts[1]
            start = int(name_parts[2])
            end = int(name_parts[3])
            chrom = str(name_parts[4])

            # Initialize list for a new chromosome
            if chrom not in transcript_intervals:
                transcript_intervals[chrom] = []

            transcript_intervals[chrom].append((start, end, strand, transcript_id))
    print("Finished building transcript map.")
    return transcript_intervals

def find_transcripts_for_locus(intervals_map, chrom, position):
    """
    Finds all transcript IDs and their strands from the map that contain a given
    genomic position.
    """
    found_transcripts = []
    # Ensure lookup key is a string to match the map keys
    chrom_key = str(chrom)

    if chrom_key in intervals_map:
        for start, end, strand, transcript_id in intervals_map[chrom_key]:
            if start <= position <= end:
                found_transcripts.append((transcript_id, strand))

    return found_transcripts

In [ ]:

# pecify input and output files
working_directory = "/content/drive/MyDrive/work/data/RNA-protein/IndelPolymorphismsModulateDistalRNAProteinInteractions"
os.chdir(working_directory)

input_path = "dataHuman/sameSites/merged.csv"
output_path = "S1_Table.csv"
fai_path = "data/transcripts.fa.fai"

# Load and process data
transcript_map = parse_fai_to_intervals(fai_path)
df = pd.read_csv(input_path)

# run annotation loop
transcript_ids_list = []
strands_list = []
for index, row in df.iterrows():
    found_results = find_transcripts_for_locus(
        transcript_map,
        row['chromosome'],
        row['indel_genomic_coordinate']
    )

    # Join multiple results with a semicolon if found, otherwise mark as Not_Found
    if found_results:
        # Separate the transcript IDs and strands into two lists
        ids = [result[0] for result in found_results]
        strands = [result[1] for result in found_results]
        transcript_ids_list.append(";".join(ids))
        strands_list.append(";".join(strands))
    else:
        transcript_ids_list.append("Not_Found")
        strands_list.append("Not_Found")

# Add the new columns to the DataFrame
df['transcript_ids'] = transcript_ids_list
#df['transcript_strands'] = strands_list

# Save Final Annotated Table
df.to_csv(output_path, index=False)
print(f"\nSuccessfully created annotated supplementary table at: {output_path}")
print("\nFirst 5 rows of the new annotated table:")
print(df.head())

Reading transcript data from: data/transcripts.fa.fai
Finished building transcript map.

Successfully created annotated supplementary table at: supplementary_table.csv

First 5 rows of the new annotated table:
  chromosome  indel_genomic_coordinate      dbSNP_ID ref_allele alt_allele  \
0          1                 225788415    rs35284825       TTTT          T   
1          1                  96821367  rs1159785954  TTATAAAGT          T   
2         17                  73225901  rs1568308614          T         TA   
3          1                 156467419   rs912337990        AGA          A   
4          9                  94176014   rs972845958        TAT          T   

         binding_site_locus      binding_region_locus  delta_delta_G  \
0  chr1:225788392-225788398  chr1:225788357-225788404      -16.28335   
1    chr1:96821390-96821396    chr1:96821390-96821412       13.21488   
2   chr17:73225873-73225879   chr17:73225854-73225879      -11.43383   
3  chr1:156467374-156467380  chr1